# Neural Collapse Dynamics: Setup and Baseline

**Before running:**
- Settings > Accelerator > **GPU P100**
- Settings > Internet > **ON**
- Click **Save & Run All** (background execution)

**Cells:** env check → CIFAR-10 → ResNet → NC metrics → train → plot

**Est. runtime: ~18 min** on P100  |  **Output:** `baseline.csv`, `fig_baseline_nc.png`

In [1]:
import torch, torchvision, sys, os, time
import torchvision.transforms as T
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

print('='*55)
print(f'PyTorch:     {torch.__version__}')
print(f'CUDA:        {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:         {torch.cuda.get_device_name(0)}')
    props = torch.cuda.get_device_properties(0)
    print(f'VRAM:        {props.total_memory/1e9:.1f} GB')
print('='*55)
torch.backends.cudnn.benchmark        = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True
DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'
SAVE_DIR = '/kaggle/working/'
print(f'Device: {DEVICE}  Save dir: {SAVE_DIR}')


PyTorch:     2.9.0+cu126
CUDA:        True
GPU:         Tesla P100-PCIE-16GB
VRAM:        17.1 GB
Device: cuda  Save dir: /kaggle/working/


/usr/local/lib/python3.12/dist-packages/torch/backends/__init__.py:46: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  self.setter(val)


In [2]:
transform_tr = T.Compose([
    T.RandomCrop(32, padding=4), T.RandomHorizontalFlip(),
    T.ToTensor(), T.Normalize((0.4914,0.4822,0.4465),(0.2023,0.1994,0.2010))])
transform_te = T.Compose([
    T.ToTensor(), T.Normalize((0.4914,0.4822,0.4465),(0.2023,0.1994,0.2010))])

trainset = torchvision.datasets.CIFAR10('/kaggle/working/data',
    train=True,  download=True, transform=transform_tr)
testset  = torchvision.datasets.CIFAR10('/kaggle/working/data',
    train=False, download=True, transform=transform_te)

train_loader = DataLoader(trainset, batch_size=128, shuffle=True,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(testset,  batch_size=256, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f'Train: {len(trainset):,} samples | {len(train_loader)} batches')
print(f'Test:  {len(testset):,}  samples | {len(test_loader)} batches')

# Speed benchmark
t0 = time.time()
for i, (x, y) in enumerate(train_loader):
    x.to(DEVICE, non_blocking=True)
    if i == 49: break
torch.cuda.synchronize()
bps = 50 / (time.time() - t0)
print(f'Loading: {bps:.1f} batches/sec')
print(f'Est. 300 epochs: ~{300*len(train_loader)/bps*3/60:.0f} min')


100%|██████████| 170M/170M [00:11<00:00, 15.4MB/s]


Train: 50,000 samples | 391 batches
Test:  10,000  samples | 40 batches
Loading: 32.4 batches/sec
Est. 300 epochs: ~181 min


In [3]:
class BasicBlock(nn.Module):
    def __init__(self, in_c, out_c, stride=1, act_cls=nn.ReLU):
        super().__init__()
        self.conv1 = nn.Conv2d(in_c, out_c, 3, stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_c)
        self.conv2 = nn.Conv2d(out_c, out_c, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_c)
        self.act   = act_cls()
        self.skip  = nn.Sequential()
        if stride != 1 or in_c != out_c:
            self.skip = nn.Sequential(
                nn.Conv2d(in_c, out_c, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_c))
    def forward(self, x):
        return self.act(self.bn2(self.conv2(self.act(self.bn1(self.conv1(x))))) + self.skip(x))

class ResNetCIFAR(nn.Module):
    # depth must be 6n+2: 20->n=3, 32->n=5, 44->n=7, 56->n=9
    def __init__(self, depth=20, act_cls=nn.ReLU, num_classes=10):
        super().__init__()
        assert (depth - 2) % 6 == 0
        n = (depth - 2) // 6
        self.conv1  = nn.Conv2d(3, 16, 3, padding=1, bias=False)
        self.bn1    = nn.BatchNorm2d(16)
        self.act1   = act_cls()
        self.layer1 = self._make(16, 16, n, 1, act_cls)
        self.layer2 = self._make(16, 32, n, 2, act_cls)
        self.layer3 = self._make(32, 64, n, 2, act_cls)
        self.pool   = nn.AdaptiveAvgPool2d(1)
        self.fc     = nn.Linear(64, num_classes)
        self._feats = None
        self.pool.register_forward_hook(
            lambda m, i, o: setattr(self, '_feats', o.flatten(1).detach()))
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def _make(self, in_c, out_c, n, stride, act_cls):
        layers = [BasicBlock(in_c, out_c, stride, act_cls)]
        for _ in range(n-1):
            layers.append(BasicBlock(out_c, out_c, 1, act_cls))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.act1(self.bn1(self.conv1(x)))
        x = self.layer3(self.layer2(self.layer1(x)))
        return self.fc(self.pool(x).flatten(1))

    def get_features(self, x):
        self(x); return self._feats

    def get_classifier_weights(self):
        return self.fc.weight.detach()

# Sanity check
for depth in [20, 32, 56]:
    m = ResNetCIFAR(depth=depth).to(DEVICE)
    x = torch.randn(4, 3, 32, 32).to(DEVICE)
    feats = m.get_features(x)
    p = sum(q.numel() for q in m.parameters()) / 1e6
    print(f'ResNet-{depth}: feats={tuple(feats.shape)}  params={p:.2f}M')


ResNet-20: feats=(4, 64)  params=0.27M
ResNet-32: feats=(4, 64)  params=0.47M
ResNet-56: feats=(4, 64)  params=0.86M


In [4]:
@torch.no_grad()
def compute_nc(model, loader, K=10):
    model.eval()
    feats_list, labels_list = [], []
    for x, y in loader:
        feats_list.append(model.get_features(x.to(DEVICE)).cpu())
        labels_list.append(y)
    H = torch.cat(feats_list).float()   # [N, d]
    Y = torch.cat(labels_list)          # [N]
    N, d = H.shape
    mu_G = H.mean(0)
    mu_c = torch.stack([H[Y==c].mean(0) for c in range(K)])
    M    = mu_c - mu_G
    # NC1
    Sw = sum((H[Y==c]-mu_c[c]).T @ (H[Y==c]-mu_c[c]) for c in range(K)) / N
    Sb = M.T @ M / K
    nc1 = (torch.trace(Sw) / torch.trace(Sb).clamp(1e-10)).item()
    # NC2 — mean |cos - (-1/(K-1))| over off-diagonal pairs
    Mn  = F.normalize(M, dim=1)
    cos = Mn @ Mn.T
    mask = ~torch.eye(K, dtype=torch.bool)
    nc2 = (cos[mask] - (-1.0/(K-1))).abs().mean().item()
    # NC3 — self-duality
    Wn  = F.normalize(model.get_classifier_weights().cpu(), dim=1)
    nc3 = (1 - (Mn * Wn).sum(1).mean()).item()
    # Feature norm
    feat_norm = H.norm(dim=1).mean().item()
    return {'nc1': nc1, 'nc2': nc2, 'nc3': nc3, 'feat_norm': feat_norm}

def evaluate(model, loader):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            correct += (model(x).argmax(1) == y).sum().item()
            total   += len(y)
    return correct / total

print('NC metrics and evaluate() ready.')


NC metrics and evaluate() ready.


In [5]:
def train_nc(model, name='model', lr=0.1, wd=1e-4, epochs=300, nc_every=5):
    model = model.to(DEVICE)
    opt   = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9,
                            weight_decay=wd, nesterov=True)
    sched = torch.optim.lr_scheduler.MultiStepLR(
                opt, milestones=[150, 225], gamma=0.1)
    rows, terminal, t0 = [], False, time.time()

    for ep in range(1, epochs+1):
        model.train()
        for x, y in train_loader:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            F.cross_entropy(model(x), y).backward()
            opt.step()
        sched.step()

        if ep % nc_every == 0 or ep == epochs:
            tr = evaluate(model, train_loader)
            te = evaluate(model, test_loader)
            if tr >= 0.99 and not terminal:
                terminal = True
                print(f'  [{name}] Terminal phase at epoch {ep}')
            nc = compute_nc(model, train_loader) if terminal else \
                 {'nc1': None, 'nc2': None, 'nc3': None, 'feat_norm': None}
            rows.append({'epoch': ep, 'train': tr, 'test': te, **nc})
            nc1s = f"{nc['nc1']:.4f}" if nc['nc1'] is not None else 'N/A'
            fns  = f"{nc['feat_norm']:.4f}" if nc['feat_norm'] else 'N/A'
            print(f'  ep={ep:>3} tr={tr:.3f} te={te:.3f} '
                  f'nc1={nc1s} fn={fns} t={(time.time()-t0)/60:.1f}m')
    return pd.DataFrame(rows)


# Baseline: ResNet-20, ReLU, wd=1e-4
print('Baseline run — ResNet-20, ReLU, wd=1e-4  (~15 min on P100)')
torch.manual_seed(0)
model_base = ResNetCIFAR(depth=20, act_cls=nn.ReLU)
df_base = train_nc(model_base, name='RN20-ReLU-1e4')
df_base.to_csv(SAVE_DIR + 'baseline.csv', index=False)

nc_done = df_base.dropna(subset=['nc1'])
if len(nc_done):
    print(f'Final NC1:  {nc_done.nc1.iloc[-1]:.6f}')
    print(f'Final NC2:  {nc_done.nc2.iloc[-1]:.6f}')
    print(f'Final NC3:  {nc_done.nc3.iloc[-1]:.6f}')
    t_nc_rows = nc_done[nc_done.nc1 < 0.1]
    if len(t_nc_rows):
        t_nc = t_nc_rows.epoch.iloc[0]
        fn   = t_nc_rows.feat_norm.iloc[0]
        print(f'T_NC:       epoch {t_nc}  feat_norm={fn:.4f}')
print(f'Test acc:   {df_base.test.iloc[-1]:.3f}')
print('Saved: baseline.csv')


Baseline run — ResNet-20, ReLU, wd=1e-4  (~15 min on P100)
  ep=  5 tr=0.799 te=0.785 nc1=N/A fn=N/A t=1.1m
  ep= 10 tr=0.839 te=0.815 nc1=N/A fn=N/A t=2.3m
  ep= 15 tr=0.852 te=0.826 nc1=N/A fn=N/A t=3.4m
  ep= 20 tr=0.880 te=0.842 nc1=N/A fn=N/A t=4.5m
  ep= 25 tr=0.876 te=0.834 nc1=N/A fn=N/A t=5.6m
  ep= 30 tr=0.877 te=0.834 nc1=N/A fn=N/A t=6.8m
  ep= 35 tr=0.901 te=0.859 nc1=N/A fn=N/A t=7.9m
  ep= 40 tr=0.890 te=0.850 nc1=N/A fn=N/A t=9.0m
  ep= 45 tr=0.874 te=0.832 nc1=N/A fn=N/A t=10.1m
  ep= 50 tr=0.890 te=0.848 nc1=N/A fn=N/A t=11.3m
  ep= 55 tr=0.911 te=0.857 nc1=N/A fn=N/A t=12.4m
  ep= 60 tr=0.898 te=0.856 nc1=N/A fn=N/A t=13.5m
  ep= 65 tr=0.916 te=0.867 nc1=N/A fn=N/A t=14.6m
  ep= 70 tr=0.897 te=0.851 nc1=N/A fn=N/A t=15.7m
  ep= 75 tr=0.926 te=0.879 nc1=N/A fn=N/A t=16.8m
  ep= 80 tr=0.920 te=0.868 nc1=N/A fn=N/A t=17.9m
  ep= 85 tr=0.921 te=0.866 nc1=N/A fn=N/A t=19.0m
  ep= 90 tr=0.902 te=0.860 nc1=N/A fn=N/A t=20.1m
  ep= 95 tr=0.921 te=0.877 nc1=N/A fn=N/A t=21.2m

In [6]:
nc_df = df_base.dropna(subset=['nc1'])
t_nc  = nc_df[nc_df.nc1 < 0.1].epoch.iloc[0] if (nc_df.nc1 < 0.1).any() else None

plt.rcParams.update({'font.family':'serif','font.size':11,
    'axes.spines.top':False,'axes.spines.right':False})
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

axes[0].plot(df_base.epoch, df_base.train, color='#2196F3', lw=2, label='Train')
axes[0].plot(df_base.epoch, df_base.test,  color='#F44336', lw=2, ls='--', label='Test')
axes[0].axhline(0.99, color='gray', ls=':', lw=1)
axes[0].set(xlabel='Epoch', ylabel='Accuracy', title='(a) Accuracy')
axes[0].legend(); axes[0].grid(alpha=0.25)

axes[1].semilogy(nc_df.epoch, nc_df.nc1, color='#4CAF50', lw=2)
axes[1].set(xlabel='Epoch', ylabel='NC1 (log)', title='(b) NC1 collapse')
if t_nc: axes[1].axvline(t_nc, color='black', ls='--', lw=1.2, label=f'T_NC={t_nc}')
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.25)

axes[2].plot(nc_df.epoch, nc_df.nc2, color='#FF9800', lw=2, label='NC2 (ETF)')
axes[2].plot(nc_df.epoch, nc_df.nc3, color='#9C27B0', lw=2, ls='--', label='NC3 (self-dual)')
axes[2].set(xlabel='Epoch', ylabel='Deviation', title='(c) NC2 and NC3')
if t_nc: axes[2].axvline(t_nc, color='black', ls='--', lw=1.2)
axes[2].legend(fontsize=9); axes[2].grid(alpha=0.25)

axes[3].plot(nc_df.epoch, nc_df.feat_norm, color='#E91E63', lw=2)
axes[3].set(xlabel='Epoch', ylabel='Mean feature norm', title='(d) Feature norm')
if t_nc:
    fn_val = nc_df[nc_df.epoch==t_nc].feat_norm.iloc[0]
    axes[3].axvline(t_nc, color='black', ls='--', lw=1.2, label=f'T_NC fn={fn_val:.3f}')
    axes[3].legend(fontsize=9)
axes[3].grid(alpha=0.25)

fig.suptitle('Baseline: ResNet-20 | ReLU | wd=1e-4 | CIFAR-10 | 300 epochs',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(SAVE_DIR + 'fig_baseline_nc.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_baseline_nc.png')
print('Setup complete. Next: run NC_Depth_Sweep.ipynb')


/tmp/ipykernel_24/2308359933.py:17: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  axes[1].legend(fontsize=9); axes[1].grid(alpha=0.25)


Saved: fig_baseline_nc.png
Setup complete. Next: run NC_Depth_Sweep.ipynb
